In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from sklearn.cluster import DBSCAN

In [2]:
INPUT_CSV  = "usa_europe_geotagged.csv"
OUT_DIR    = "poi_clusters"
CHUNKSIZE  = 10_000

# eps in radians = 100m / 6371000 (earth radius in meters)
EPS        = 100 / 6_371_000
MIN_SAMPLES = 3   # minimum 3 photos to form a POI

os.makedirs(OUT_DIR, exist_ok=True)
print("Config ready.")
print(f"  EPS (100m in radians) : {EPS:.8f}")
print(f"  Min samples per POI   : {MIN_SAMPLES}")

Config ready.
  EPS (100m in radians) : 0.00001570
  Min samples per POI   : 3


In [3]:
df_head = pd.read_csv(INPUT_CSV, nrows=5)
print("Columns:", df_head.columns.tolist())
print()
df_head[['photoid', 'latitude', 'longitude']]

Columns: ['photoid', 'uid', 'unickname', 'datetaken', 'dateuploaded', 'capturedevice', 'title', 'description', 'usertags', 'machinetags', 'longitude', 'latitude', 'accuracy', 'pageurl', 'downloadurl', 'licensename', 'licenseurl', 'serverid', 'farmid', 'secret', 'secretoriginal', 'ext', 'marker']



,photoid,latitude,longitude
0,29872,38.760236,-93.440796
1,29873,38.760236,-93.440796
2,29874,38.760236,-93.440796
3,29875,38.760236,-93.440796
4,30431,47.619673,-122.351903


In [4]:
print("Loading photoid, lat, lon from CSV...")

df = pd.read_csv(
    INPUT_CSV,
    usecols=['photoid', 'latitude', 'longitude'],
    dtype={'photoid': np.int64, 'latitude': np.float32, 'longitude': np.float32}
)

df = df.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

print(f"Total rows loaded : {len(df):,}")
print(f"Memory usage      : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(df.head())

Loading photoid, lat, lon from CSV...
Total rows loaded : 7,672,417
Memory usage      : 122.8 MB
   photoid   longitude   latitude
0    29872  -93.440796  38.760235
1    29873  -93.440796  38.760235
2    29874  -93.440796  38.760235
3    29875  -93.440796  38.760235
4    30431 -122.351906  47.619675


In [5]:
coords = np.radians(
    df[['latitude', 'longitude']].values.astype(np.float64)
)

print(f"Coords shape : {coords.shape}")
print(f"Sample row   : {coords[0]}")

Coords shape : (7672417, 2)
Sample row   : [ 0.67649372 -1.63084954]


In [6]:
db = DBSCAN(
    eps=EPS,
    min_samples=MIN_SAMPLES,
    algorithm='ball_tree',
    metric='haversine',
    n_jobs=-1
).fit(coords)

labels = db.labels_

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = (labels == -1).sum()

print(f"POIs found        : {n_clusters:,}")
print(f"Noise points      : {n_noise:,}")
print(f"Clustered photos  : {len(labels) - n_noise:,}")
print(f"Clustering rate   : {(len(labels) - n_noise) / len(labels) * 100:.1f}%")

POIs found        : 239,046
Noise points      : 750,590
Clustered photos  : 6,921,827
Clustering rate   : 90.2%


In [7]:
photoids = df['photoid'].values.astype(np.int64)
labels   = labels.astype(np.int32)

np.save(os.path.join(OUT_DIR, "photoid_to_poi.npy"), photoids)
np.save(os.path.join(OUT_DIR, "poi_labels.npy"),     labels)

print(f"Saved → {OUT_DIR}/photoid_to_poi.npy")
print(f"Saved → {OUT_DIR}/poi_labels.npy")
print(f"Shape : {photoids.shape}")

Saved → poi_clusters/photoid_to_poi.npy
Saved → poi_clusters/poi_labels.npy
Shape : (7672417,)


In [8]:
photoids = np.load(os.path.join(OUT_DIR, "photoid_to_poi.npy"))
labels   = np.load(os.path.join(OUT_DIR, "poi_labels.npy"))

print(f"Photoids shape   : {photoids.shape}")
print(f"Labels shape     : {labels.shape}")
print(f"Sample photoids  : {photoids[:5]}")
print(f"Sample labels    : {labels[:5]}")
print(f"Unique POIs      : {len(np.unique(labels[labels != -1])):,}")
print(f"Noise points (-1): {(labels == -1).sum():,}")

Photoids shape   : (7672417,)
Labels shape     : (7672417,)
Sample photoids  : [29872 29873 29874 29875 30431]
Sample labels    : [0 0 0 0 1]
Unique POIs      : 239,046
Noise points (-1): 750,590
